# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saibalaji2004/Flyrank-intern-repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [2]:

!git clone https://github.com/saibalaji2004/Flyrank-intern-repo.git


Cloning into 'Flyrank-intern-repo'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 137 (delta 46), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 1.88 MiB | 8.27 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [3]:

from pathlib import Path

REPO = Path("/content/Flyrank-intern-repo")
DATA_PATH = REPO / "data/raw/content_refresh_anonymized.csv"

print("Repository exists:", REPO.exists())
print("Dataset exists:", DATA_PATH.exists())

if REPO.exists():
    print("\nRepository folders:")
    for path in REPO.iterdir():
        print(path.name)


Repository exists: True
Dataset exists: True

Repository folders:
docs
.github
DATA_USE.md
CLAUDE.md
LICENSE
.gitignore
AGENTS.md
README.md
notebooks
submission
SETUP.md
requirements.txt
skills
data
work
GUIDE.md
.git
scripts
outputs


In [4]:

from pathlib import Path
import pandas as pd

REPO = Path("/content/Flyrank-intern-repo")
DATA_PATH = REPO / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumn names:")
for column in df.columns:
    print(column)

print("\nMissing values in columns:")
display(df.isnull().sum().sort_values(ascending=False).head(15).to_frame("missing_count"))

print("\nFirst 5 rows:")
display(df.head())


Dataset shape: (30000, 44)

Column names:
content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct

Missing values in columns:


,missing_count
provider_used,21438
word_count,7699
char_count,7699
word_count_tier,7699
char_count_tier,7699
model_used,5733
trend_pct,3388
competition_level,2610
search_volume,2468
cpc,2468



First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [5]:

import numpy as np
import pandas as pd

# Work only with the repository dataset.
df = pd.read_csv(DATA_PATH).copy()

# --------------------------------------------------
# SIGNAL CHECK 1: Staleness vs declining pages
# trend_direction is used ONLY as an outcome here,
# never as a scoring feature.
# --------------------------------------------------

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 90, 180, 365, np.inf],
    labels=["0-90 days", "91-180 days", "181-365 days", "Over 365 days"]
)

df["is_declining_for_check"] = (
    df["trend_direction"].astype("string").str.lower().eq("down")
    & df["trend_direction"].notna()
)

staleness_table = (
    df.dropna(subset=["staleness_bucket"])
      .groupby("staleness_bucket", observed=True)
      .agg(
          n=("content_id", "size"),
          declining_pages=("is_declining_for_check", "sum"),
          decline_rate=("is_declining_for_check", "mean")
      )
      .reset_index()
)

staleness_table["decline_rate_pct"] = (
    staleness_table["decline_rate"] * 100
).round(2)

print("SIGNAL CHECK 1: STALENESS")
display(staleness_table[
    ["staleness_bucket", "n", "declining_pages", "decline_rate_pct"]
])

valid_stale = staleness_table[staleness_table["n"] >= 30]

if len(valid_stale) < 2:
    stale_verdict = "MIXED"
else:
    fresh_rate = valid_stale.iloc[0]["decline_rate"]
    stale_rate = valid_stale.iloc[-1]["decline_rate"]
    difference_pp = (stale_rate - fresh_rate) * 100

    if difference_pp >= 5:
        stale_verdict = "CONFIRMED"
    elif difference_pp <= -5:
        stale_verdict = "OPPOSITE"
    else:
        stale_verdict = "MIXED"

print("Staleness verdict:", stale_verdict)
print("Difference between oldest and newest bucket (percentage points):",
      round(difference_pp, 2) if len(valid_stale) >= 2 else "Not enough data")


# --------------------------------------------------
# SIGNAL CHECK 2: Search position vs CTR
# Check whether CTR generally decreases as ranking
# position gets worse.
# --------------------------------------------------

df["position_bucket"] = pd.cut(
    df["avg_position"].where(df["avg_position"] > 0),
    bins=[0, 3, 10, 20, 100, np.inf],
    labels=["1-3", "4-10", "11-20", "21-100", "Over 100"]
)

ctr_position_table = (
    df.dropna(subset=["position_bucket", "ctr"])
      .groupby("position_bucket", observed=True)
      .agg(
          n=("content_id", "size"),
          median_ctr=("ctr", "median"),
          mean_ctr=("ctr", "mean")
      )
      .reset_index()
)

ctr_position_table[["median_ctr", "mean_ctr"]] = (
    ctr_position_table[["median_ctr", "mean_ctr"]].round(3)
)

print("\nSIGNAL CHECK 2: SEARCH POSITION VS CTR")
display(ctr_position_table)

medians = ctr_position_table["median_ctr"].dropna().tolist()

if len(medians) < 3:
    ctr_verdict = "MIXED"
elif all(medians[i] >= medians[i + 1] for i in range(len(medians) - 1)):
    ctr_verdict = "CONFIRMED"
elif all(medians[i] <= medians[i + 1] for i in range(len(medians) - 1)):
    ctr_verdict = "OPPOSITE"
else:
    ctr_verdict = "MIXED"

print("CTR vs position verdict:", ctr_verdict)


SIGNAL CHECK 1: STALENESS


,staleness_bucket,n,declining_pages,decline_rate_pct
0,0-90 days,20655,10576,51.2
1,91-180 days,9171,5604,61.11
2,181-365 days,169,79,46.75
3,Over 365 days,5,3,60.0


Staleness verdict: MIXED
Difference between oldest and newest bucket (percentage points): -4.46

SIGNAL CHECK 2: SEARCH POSITION VS CTR


,position_bucket,n,median_ctr,mean_ctr
0,1-3,1141,0.00,2.714
1,4-10,11842,0.16,0.651
2,11-20,7273,0.10,0.323
3,21-100,8524,0.00,0.212
4,Over 100,15,0.00,0.000


CTR vs position verdict: MIXED



## Baseline Rule and Reason Codes

Prioritize pages for manual review using two signals: content staleness and search visibility.

- Stale: the page has not been updated for at least 90 days.
- Visible: the page's 90-day impressions are at or above the median among pages with positive impressions.
- Score: 2 points for staleness and 1 point for visibility.

### Actions
- Score 3: REVIEW_CONTENT
- Score 2: REVIEW_STALENESS
- Score 1: REVIEW_VISIBILITY
- Score 0: MONITOR

### Reason codes
- STALE_AND_VISIBLE
- STALE_ONLY
- VISIBLE_ONLY
- NEITHER_SIGNAL

Each page receives exactly one reason code. This is a transparent prioritization baseline, not a trained prediction model. The signal checks were mixed, so the rule must be evaluated rather than assumed effective.


In [6]:

import numpy as np
import pandas as pd
from pathlib import Path

# Use the dataset and repository paths already established.
REPO = Path("/content/Flyrank-intern-repo")
DATA_PATH = REPO / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

# Prepare the two scoring signals.
df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"], errors="coerce"
)
df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"], errors="coerce"
)

positive_impressions = df.loc[
    df["impressions_90d"] > 0, "impressions_90d"
].dropna()

if positive_impressions.empty:
    raise ValueError("No positive impression values found.")

visibility_threshold = positive_impressions.median()

df["is_stale"] = df["days_since_last_update"].ge(90).fillna(False)
df["is_visible"] = df["impressions_90d"].ge(
    visibility_threshold
).fillna(False)

# Transparent score.
df["score"] = (
    2 * df["is_stale"].astype(int)
    + df["is_visible"].astype(int)
)

# One reason code and one action per page.
reason_map = {
    (True, True): "STALE_AND_VISIBLE",
    (True, False): "STALE_ONLY",
    (False, True): "VISIBLE_ONLY",
    (False, False): "NEITHER_SIGNAL",
}

action_map = {
    (True, True): "REVIEW_CONTENT",
    (True, False): "REVIEW_STALENESS",
    (False, True): "REVIEW_VISIBILITY",
    (False, False): "MONITOR",
}

df["reason_code"] = [
    reason_map[(stale, visible)]
    for stale, visible in zip(df["is_stale"], df["is_visible"])
]

df["action_label"] = [
    action_map[(stale, visible)]
    for stale, visible in zip(df["is_stale"], df["is_visible"])
]

# Rank pages by score, then staleness, then impressions.
df["days_rank"] = df["days_since_last_update"].fillna(-1)
df["impressions_rank"] = df["impressions_90d"].fillna(-1)

ranked = df.sort_values(
    ["score", "days_rank", "impressions_rank"],
    ascending=[False, False, False],
    kind="mergesort"
).reset_index(drop=True)

ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))

# Save the CSV to the required repository output folder.
OUTPUT_PATH = REPO / "work/outputs/baseline_action_score.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

output_columns = [
    "rank",
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "score",
    "reason_code",
    "action_label",
]

ranked[output_columns].to_csv(OUTPUT_PATH, index=False)

print("Visibility threshold:", visibility_threshold)
print("Rows ranked:", len(ranked))
print("CSV saved:", OUTPUT_PATH)

print("\nAction counts:")
display(
    ranked["action_label"]
    .value_counts()
    .rename_axis("action_label")
    .to_frame("n")
)

print("\nTop 10 ranked pages:")
display(ranked[output_columns].head(10))

print("\nTop 20 ranked pages:")
display(ranked[output_columns].head(20))


Visibility threshold: 731.0
Rows ranked: 30000
CSV saved: /content/Flyrank-intern-repo/work/outputs/baseline_action_score.csv

Action counts:


,n
action_label,
MONITOR,11640
REVIEW_VISIBILITY,9015
REVIEW_CONTENT,5992
REVIEW_STALENESS,3353



Top 10 ranked pages:


,rank,content_id,client_id,days_since_last_update,impressions_90d,score,reason_code,action_label
0,1,content_7f116ae1f6f5,client_9400f1b21c,301,954,3,STALE_AND_VISIBLE,REVIEW_CONTENT
1,2,content_72496874f806,client_4ec9599fc2,301,821,3,STALE_AND_VISIBLE,REVIEW_CONTENT
2,3,content_cf56e2e2e282,client_7f2253d7e2,194,61678,3,STALE_AND_VISIBLE,REVIEW_CONTENT
3,4,content_7368877ea310,client_7f2253d7e2,194,59472,3,STALE_AND_VISIBLE,REVIEW_CONTENT
4,5,content_1bfaa38ff26c,client_7f2253d7e2,194,25715,3,STALE_AND_VISIBLE,REVIEW_CONTENT
5,6,content_5feee3994adb,client_7f2253d7e2,194,7812,3,STALE_AND_VISIBLE,REVIEW_CONTENT
6,7,content_b16bd7307b39,client_7f2253d7e2,194,4590,3,STALE_AND_VISIBLE,REVIEW_CONTENT
7,8,content_fe16a55cd13d,client_7f2253d7e2,194,4556,3,STALE_AND_VISIBLE,REVIEW_CONTENT
8,9,content_ecb6215e79fd,client_7f2253d7e2,194,4429,3,STALE_AND_VISIBLE,REVIEW_CONTENT
9,10,content_bdbec75c1148,client_7f2253d7e2,194,1316,3,STALE_AND_VISIBLE,REVIEW_CONTENT



Top 20 ranked pages:


,rank,content_id,client_id,days_since_last_update,impressions_90d,score,reason_code,action_label
0,1,content_7f116ae1f6f5,client_9400f1b21c,301,954,3,STALE_AND_VISIBLE,REVIEW_CONTENT
1,2,content_72496874f806,client_4ec9599fc2,301,821,3,STALE_AND_VISIBLE,REVIEW_CONTENT
2,3,content_cf56e2e2e282,client_7f2253d7e2,194,61678,3,STALE_AND_VISIBLE,REVIEW_CONTENT
3,4,content_7368877ea310,client_7f2253d7e2,194,59472,3,STALE_AND_VISIBLE,REVIEW_CONTENT
4,5,content_1bfaa38ff26c,client_7f2253d7e2,194,25715,3,STALE_AND_VISIBLE,REVIEW_CONTENT
5,6,content_5feee3994adb,client_7f2253d7e2,194,7812,3,STALE_AND_VISIBLE,REVIEW_CONTENT
6,7,content_b16bd7307b39,client_7f2253d7e2,194,4590,3,STALE_AND_VISIBLE,REVIEW_CONTENT
7,8,content_fe16a55cd13d,client_7f2253d7e2,194,4556,3,STALE_AND_VISIBLE,REVIEW_CONTENT
8,9,content_ecb6215e79fd,client_7f2253d7e2,194,4429,3,STALE_AND_VISIBLE,REVIEW_CONTENT
9,10,content_bdbec75c1148,client_7f2253d7e2,194,1316,3,STALE_AND_VISIBLE,REVIEW_CONTENT


In [7]:

# Evaluate the ranked queue.
# trend_direction is used ONLY for evaluation, never for scoring.

evaluation = ranked.copy()

evaluation["decline_outcome"] = (
    evaluation["trend_direction"]
    .astype("string")
    .str.lower()
    .eq("down")
    & evaluation["trend_direction"].notna()
)

base_rate = evaluation["decline_outcome"].mean()

print("Overall decline base rate:", round(base_rate * 100, 2), "%")

for k in [10, 20]:
    top_k = evaluation.head(k)
    precision_k = top_k["decline_outcome"].mean()

    print(
        f"Precision@{k}: {precision_k:.3f} "
        f"({precision_k * 100:.2f}%)"
    )

print("\nTop-20 evaluation table:")
display(
    evaluation.head(20)[
        [
            "rank",
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "score",
            "reason_code",
            "action_label",
            "decline_outcome",
        ]
    ]
)


Overall decline base rate: 54.21 %
Precision@10: 0.900 (90.00%)
Precision@20: 0.700 (70.00%)

Top-20 evaluation table:


,rank,content_id,days_since_last_update,impressions_90d,score,reason_code,action_label,decline_outcome
0,1,content_7f116ae1f6f5,301,954,3,STALE_AND_VISIBLE,REVIEW_CONTENT,True
1,2,content_72496874f806,301,821,3,STALE_AND_VISIBLE,REVIEW_CONTENT,True
2,3,content_cf56e2e2e282,194,61678,3,STALE_AND_VISIBLE,REVIEW_CONTENT,True
3,4,content_7368877ea310,194,59472,3,STALE_AND_VISIBLE,REVIEW_CONTENT,True
4,5,content_1bfaa38ff26c,194,25715,3,STALE_AND_VISIBLE,REVIEW_CONTENT,True
5,6,content_5feee3994adb,194,7812,3,STALE_AND_VISIBLE,REVIEW_CONTENT,True
6,7,content_b16bd7307b39,194,4590,3,STALE_AND_VISIBLE,REVIEW_CONTENT,True
7,8,content_fe16a55cd13d,194,4556,3,STALE_AND_VISIBLE,REVIEW_CONTENT,True
8,9,content_ecb6215e79fd,194,4429,3,STALE_AND_VISIBLE,REVIEW_CONTENT,True
9,10,content_bdbec75c1148,194,1316,3,STALE_AND_VISIBLE,REVIEW_CONTENT,False


In [8]:

# ML-07: Top-20 manual review worksheet

review = ranked.head(20).copy()

def explain_rank(row):
    if row["reason_code"] == "STALE_AND_VISIBLE":
        return (
            f"Not updated for {int(row['days_since_last_update'])} days; "
            f"received {int(row['impressions_90d'])} impressions in 90 days."
        )
    if row["reason_code"] == "STALE_ONLY":
        return "Stale page, but it does not meet the visibility threshold."
    if row["reason_code"] == "VISIBLE_ONLY":
        return "Visible in search, but it does not meet the staleness threshold."
    return "Neither staleness nor visibility threshold was met."

review["why_ranked"] = review.apply(explain_rank, axis=1)

review["confidence_note"] = (
    "Moderate confidence in the rule-based priority; "
    "the two signal checks were mixed."
)

review["what_could_make_it_wrong"] = (
    "The page may already be accurate and useful; "
    "impressions and staleness alone do not prove a content problem."
)

review["manual_review_decision"] = "Inspect page before making changes"

review_columns = [
    "rank",
    "content_id",
    "action_label",
    "reason_code",
    "why_ranked",
    "confidence_note",
    "what_could_make_it_wrong",
    "manual_review_decision",
]

display(review[review_columns])

# Save the review worksheet locally for use during manual review.
REVIEW_PATH = REPO / "work/outputs/top20_manual_review.csv"
review[review_columns].to_csv(REVIEW_PATH, index=False)

print("Review worksheet saved to:", REVIEW_PATH)


,rank,content_id,action_label,reason_code,why_ranked,confidence_note,what_could_make_it_wrong,manual_review_decision
0,1,content_7f116ae1f6f5,REVIEW_CONTENT,STALE_AND_VISIBLE,Not updated for 301 days; received 954 impress...,Moderate confidence in the rule-based priority...,The page may already be accurate and useful; i...,Inspect page before making changes
1,2,content_72496874f806,REVIEW_CONTENT,STALE_AND_VISIBLE,Not updated for 301 days; received 821 impress...,Moderate confidence in the rule-based priority...,The page may already be accurate and useful; i...,Inspect page before making changes
2,3,content_cf56e2e2e282,REVIEW_CONTENT,STALE_AND_VISIBLE,Not updated for 194 days; received 61678 impre...,Moderate confidence in the rule-based priority...,The page may already be accurate and useful; i...,Inspect page before making changes
3,4,content_7368877ea310,REVIEW_CONTENT,STALE_AND_VISIBLE,Not updated for 194 days; received 59472 impre...,Moderate confidence in the rule-based priority...,The page may already be accurate and useful; i...,Inspect page before making changes
4,5,content_1bfaa38ff26c,REVIEW_CONTENT,STALE_AND_VISIBLE,Not updated for 194 days; received 25715 impre...,Moderate confidence in the rule-based priority...,The page may already be accurate and useful; i...,Inspect page before making changes
5,6,content_5feee3994adb,REVIEW_CONTENT,STALE_AND_VISIBLE,Not updated for 194 days; received 7812 impres...,Moderate confidence in the rule-based priority...,The page may already be accurate and useful; i...,Inspect page before making changes
6,7,content_b16bd7307b39,REVIEW_CONTENT,STALE_AND_VISIBLE,Not updated for 194 days; received 4590 impres...,Moderate confidence in the rule-based priority...,The page may already be accurate and useful; i...,Inspect page before making changes
7,8,content_fe16a55cd13d,REVIEW_CONTENT,STALE_AND_VISIBLE,Not updated for 194 days; received 4556 impres...,Moderate confidence in the rule-based priority...,The page may already be accurate and useful; i...,Inspect page before making changes
8,9,content_ecb6215e79fd,REVIEW_CONTENT,STALE_AND_VISIBLE,Not updated for 194 days; received 4429 impres...,Moderate confidence in the rule-based priority...,The page may already be accurate and useful; i...,Inspect page before making changes
9,10,content_bdbec75c1148,REVIEW_CONTENT,STALE_AND_VISIBLE,Not updated for 194 days; received 1316 impres...,Moderate confidence in the rule-based priority...,The page may already be accurate and useful; i...,Inspect page before making changes


Review worksheet saved to: /content/Flyrank-intern-repo/work/outputs/top20_manual_review.csv


In [9]:

# Identify top-20 pages that did not have a declining outcome.
# This is evaluation analysis only, not part of the scoring rule.

top20_eval = evaluation.head(20).copy()

weak_picks = top20_eval.loc[
    ~top20_eval["decline_outcome"],
    [
        "rank",
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "score",
        "reason_code",
        "action_label",
    ],
]

print("Weak picks in the top 20:", len(weak_picks))
display(weak_picks)

# Leakage checks: these fields must not be scoring inputs.
scoring_features = {
    "days_since_last_update",
    "impressions_90d",
}

forbidden_features = {
    "trend_direction",
    "trend_pct",
    "decline_outcome",
    "is_declining_label",
}

print("\nScoring features:", sorted(scoring_features))
print("Forbidden outcome-derived fields:", sorted(forbidden_features))
print(
    "Leakage check:",
    "PASS" if scoring_features.isdisjoint(forbidden_features) else "FAIL"
)

# Check that each page has exactly one non-empty reason code and action.
print(
    "Every row has one reason code:",
    ranked["reason_code"].notna().all()
)
print(
    "Every row has one action label:",
    ranked["action_label"].notna().all()
)


Weak picks in the top 20: 6


,rank,content_id,days_since_last_update,impressions_90d,score,reason_code,action_label
9,10,content_bdbec75c1148,194,1316,3,STALE_AND_VISIBLE,REVIEW_CONTENT
15,16,content_cb7e312f5d32,151,21272,3,STALE_AND_VISIBLE,REVIEW_CONTENT
16,17,content_2f38c656fc95,151,1830,3,STALE_AND_VISIBLE,REVIEW_CONTENT
17,18,content_6c56c0675297,151,1790,3,STALE_AND_VISIBLE,REVIEW_CONTENT
18,19,content_4df0b7207fe3,151,1700,3,STALE_AND_VISIBLE,REVIEW_CONTENT
19,20,content_8a80ae32342f,151,751,3,STALE_AND_VISIBLE,REVIEW_CONTENT



Scoring features: ['days_since_last_update', 'impressions_90d']
Forbidden outcome-derived fields: ['decline_outcome', 'is_declining_label', 'trend_direction', 'trend_pct']
Leakage check: PASS
Every row has one reason code: True
Every row has one action label: True


In [10]:

from pathlib import Path

REPO = Path("/content/Flyrank-intern-repo")

required_files = [
    REPO / "work/notebooks/w04_baseline_score.ipynb",
    REPO / "work/outputs/baseline_action_score.csv",
    REPO / "work/outputs/top20_manual_review.csv",
]

for file in required_files:
    print(f"{'FOUND' if file.exists() else 'MISSING'}: {file}")


FOUND: /content/Flyrank-intern-repo/work/notebooks/w04_baseline_score.ipynb
FOUND: /content/Flyrank-intern-repo/work/outputs/baseline_action_score.csv
FOUND: /content/Flyrank-intern-repo/work/outputs/top20_manual_review.csv


In [11]:

import subprocess
from pathlib import Path

REPO = Path("/content/Flyrank-intern-repo")

def run_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=REPO,
        capture_output=True,
        text=True
    )
    print("$ git", " ".join(args))
    print(result.stdout or result.stderr or "(no output)")
    return result

print("1. Git status:")
run_git("status", "--short")

print("\n2. Check whether the CSV files are ignored:")
for filename in [
    "work/outputs/baseline_action_score.csv",
    "work/outputs/top20_manual_review.csv",
]:
    result = run_git("check-ignore", "-v", filename)
    if result.returncode != 0:
        print("WARNING: This file is NOT confirmed as ignored:", filename)

print("\n3. Check whether the notebook is tracked:")
run_git("ls-files", "work/notebooks/w04_baseline_score.ipynb")


1. Git status:
$ git status --short
(no output)

2. Check whether the CSV files are ignored:
$ git check-ignore -v work/outputs/baseline_action_score.csv
.gitignore:28:work/**/*.csv	work/outputs/baseline_action_score.csv

$ git check-ignore -v work/outputs/top20_manual_review.csv
.gitignore:28:work/**/*.csv	work/outputs/top20_manual_review.csv


3. Check whether the notebook is tracked:
$ git ls-files work/notebooks/w04_baseline_score.ipynb
work/notebooks/w04_baseline_score.ipynb



CompletedProcess(args=['git', 'ls-files', 'work/notebooks/w04_baseline_score.ipynb'], returncode=0, stdout='work/notebooks/w04_baseline_score.ipynb\n', stderr='')